# Notebook 2: Recipe Knowledge Base
**Fridge Hero — BT5153 Group 9**

This notebook builds:
1. `recipe_clean.csv` — preprocessed recipe dataset
2. ChromaDB collection — SBERT-embedded recipes for semantic retrieval
3. `substitution.json` — ingredient substitution knowledge base (LLM-generated)

**Model used:** `all-MiniLM-L6-v2` (SBERT) — fast, lightweight, good quality

## 0. Setup

In [1]:
# Install if needed:
# pip install sentence-transformers chromadb tqdm requests

import pandas as pd
import numpy as np
import ast, json, re, os
from tqdm import tqdm
import requests
import warnings
warnings.filterwarnings('ignore')
import json
from collections import defaultdict

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

DATA_DIR  = './data/'
OUT_DIR   = './output/'
CHROMA_DIR = './chromadb/'
os.makedirs(OUT_DIR,    exist_ok=True)
os.makedirs(CHROMA_DIR, exist_ok=True)

OLLAMA_URL   = 'http://localhost:11434/api/generate'
OLLAMA_MODEL = 'qwen2.5:7b'

SBERT_MODEL  = 'all-MiniLM-L6-v2'   # 384-dim, fast
BATCH_SIZE   = 512                   # embedding batch size
MAX_RECIPES  = None                  # set to e.g. 50000 for faster dev run

print('Setup complete.')

Setup complete.


In [2]:
import requests
try:
    r = requests.get('http://localhost:11434/api/tags', timeout=5)
    print('Ollama running:', [m['name'] for m in r.json().get('models',[])])
except:
    print('Ollama NOT running — start with: ollama serve')

Ollama running: ['qwen2.5:7b', 'llama3.2:latest', 'nomic-embed-text:latest', 'qwen3:4b', 'qwen2.5:3b', 'gpt-oss:120b-cloud']


## 1. Load & Clean Recipes

In [3]:
recipes_raw = pd.read_csv(DATA_DIR + 'RAW_recipes.csv', nrows=MAX_RECIPES)
print('Raw shape:', recipes_raw.shape)
recipes_raw.head(2)

Raw shape: (231637, 12)


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6


In [4]:
def safe_parse_list(val):
    if isinstance(val, list): return val
    if pd.isna(val): return []
    try: return ast.literal_eval(val)
    except: return []

def parse_nutrition(val):
    keys = ['calories','total_fat_pdv','sugar_pdv','sodium_pdv',
            'protein_pdv','sat_fat_pdv','carbs_pdv']
    lst = safe_parse_list(val)
    return dict(zip(keys, lst)) if len(lst) == 7 else {k: np.nan for k in keys}

df = recipes_raw.copy()
for col in ['ingredients','steps','tags']:
    df[col] = df[col].apply(safe_parse_list)

# parse nutrition into columns
nutr = df['nutrition'].apply(parse_nutrition).apply(pd.Series)
df = pd.concat([df.drop(columns=['nutrition']), nutr], axis=1)

# ── Cleaning ──────────────────────────────────────────────────────────────────
# remove extreme cook times (> 24 hrs likely data error)
df = df[df['minutes'] <= 1440].copy()

# remove recipes with no ingredients
df = df[df['ingredients'].apply(len) > 0].copy()

# remove calorie outliers (IQR)
q1, q3 = df['calories'].quantile([0.01, 0.99])
df = df[(df['calories'] >= q1) & (df['calories'] <= q3)].copy()

df = df.reset_index(drop=True)
print('Clean shape:', df.shape)

Clean shape: (225044, 18)


## 2. Feature Engineering for Retrieval

In [5]:
# flat string fields
df['ingredients_str'] = df['ingredients'].apply(lambda x: ', '.join(x))
df['tags_str']        = df['tags'].apply(lambda x: ', '.join(x))
df['steps_str']       = df['steps'].apply(lambda x: ' '.join(x))

# ── Chunk text for embedding ───────────────────────────────────────────────────
# We embed a rich text chunk per recipe combining name + tags + ingredients
# Steps are stored separately and only shown in output, not embedded
def make_embed_text(row):
    return (
        f"Recipe: {row['name']}. "
        f"Tags: {row['tags_str']}. "
        f"Ingredients: {row['ingredients_str']}. "
        f"Cook time: {int(row['minutes'])} minutes. "
        f"Steps: {int(row['n_steps'])}."
    )

df['embed_text'] = df.apply(make_embed_text, axis=1)

# dietary convenience flags (rule-based from tags)
df['is_vegetarian'] = df['tags_str'].str.contains('vegetarian', case=False, na=False)
df['is_vegan']      = df['tags_str'].str.contains('vegan', case=False, na=False)
df['is_spicy']      = df['tags_str'].str.contains('spicy|hot-and-spicy', case=False, na=False)
df['is_low_fat']    = df['tags_str'].str.contains('low-fat|low-in-something', case=False, na=False)

# difficulty proxy
df['difficulty'] = pd.cut(df['n_steps'],
                           bins=[0,5,10,20,np.inf],
                           labels=['easy','medium','hard','very_hard'])

# save clean recipe CSV
df.to_csv(OUT_DIR + 'recipe_clean.csv', index=False)
print('Saved recipe_clean.csv')
df[['id','name','minutes','calories','difficulty','is_vegetarian','is_spicy']].head(5)

Saved recipe_clean.csv


,id,name,minutes,calories,difficulty,is_vegetarian,is_spicy
0,137739,arriba baked winter squash mexican style,55,51.5,hard,True,False
1,31490,a bit different breakfast pizza,30,173.4,medium,False,False
2,112140,all in the kitchen chili,130,269.8,medium,False,False
3,59389,alouette potatoes,45,368.1,hard,False,False
4,44061,amish tomato ketchup for canning,190,352.9,easy,True,False


## 3. SBERT Embedding (all-MiniLM-L6-v2)

In [6]:
print(f'Loading SBERT model: {SBERT_MODEL}')
sbert = SentenceTransformer(SBERT_MODEL)
print(f'Embedding dimension: {sbert.get_sentence_embedding_dimension()}')

Loading SBERT model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dimension: 384


In [7]:
texts = df['embed_text'].tolist()

print(f'Encoding {len(texts):,} recipes in batches of {BATCH_SIZE}...')
embeddings = sbert.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True   # cosine similarity ready
)
print('Embeddings shape:', embeddings.shape)

Encoding 225,044 recipes in batches of 512...


Batches:   0%|          | 0/440 [00:00<?, ?it/s]

Embeddings shape: (225044, 384)


## 4. Store in ChromaDB

In [8]:
client = chromadb.PersistentClient(path=CHROMA_DIR)

# delete existing collection if re-running
try:
    client.delete_collection('recipes')
    print('Deleted existing collection.')
except:
    pass

collection = client.create_collection(
    name='recipes',
    metadata={'hnsw:space': 'cosine'}   # cosine similarity
)

print('Created ChromaDB collection: recipes')

Deleted existing collection.
Created ChromaDB collection: recipes


In [9]:
# add in batches (ChromaDB has per-call limits)
CHROMA_BATCH = 5000

for start in tqdm(range(0, len(df), CHROMA_BATCH)):
    end   = min(start + CHROMA_BATCH, len(df))
    batch = df.iloc[start:end]

    collection.add(
        ids        = batch['id'].astype(str).tolist(),
        embeddings = embeddings[start:end].tolist(),
        documents  = batch['embed_text'].tolist(),
        metadatas  = [
            {
                'name':          str(r['name']),
                'minutes':       int(r['minutes']),
                'n_steps':       int(r['n_steps']),
                'calories':      float(round(r['calories'], 1)),
                'protein_pdv':   float(round(r['protein_pdv'], 1)),
                'is_vegetarian': bool(r['is_vegetarian']),
                'is_vegan':      bool(r['is_vegan']),
                'is_spicy':      bool(r['is_spicy']),
                'difficulty':    str(r['difficulty']),
                'ingredients_str': str(r['ingredients_str']),
                'tags_str':      str(r['tags_str']),
                'n_ingredients': int(len([x for x in str(r['ingredients_str']).split(',') if x.strip()])),
            }
            for _, r in batch.iterrows()
        ]
    )

print(f'ChromaDB collection size: {collection.count():,}')

100%|██████████| 46/46 [04:52<00:00,  6.36s/it]

ChromaDB collection size: 225,044


In [10]:
# ── Quick retrieval test ───────────────────────────────────────────────────────
test_query = 'spicy Thai chicken with basil'
q_embed = sbert.encode([test_query], normalize_embeddings=True)

results = collection.query(
    query_embeddings=q_embed.tolist(),
    n_results=5
)

print(f'Query: "{test_query}"')
print('Top-5 results:')
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0],
    results['metadatas'][0],
    results['distances'][0]
)):
    print(f"  {i+1}. {meta['name']} | {meta['minutes']}min | sim={1-dist:.3f}")

Query: "spicy Thai chicken with basil"
Top-5 results:
  1. spicy thai chicken with basil | 25min | sim=0.832
  2. thai style chicken with basil  cook s illustrated | 40min | sim=0.786
  3. thai spicy basil chicken fried rice | 45min | sim=0.782
  4. spicy basil chicken | 40min | sim=0.777
  5. spicy thai style chicken | 30min | sim=0.773


## 5. Build Substitution Knowledge Base

In [13]:
# Load from Food.com review substitution data
with open(DATA_DIR + 'fooddotcom_review_sub_data_clean.json') as f:
    raw = json.load(f)

sub_db = defaultdict(list)
for row in raw:
    frm = row['fromIng'].strip().lower()
    to  = row['toIng'].strip().lower()
    if frm and to and to not in sub_db[frm]:
        sub_db[frm].append(to)

with open(OUT_DIR + 'substitution.json', 'w') as f:
    json.dump(dict(sub_db), f, indent=2)

print(f'Ingredients with substitutes: {len(sub_db)}')
print(f'Total substitution pairs    : {sum(len(v) for v in sub_db.values())}')

Ingredients with substitutes: 1385
Total substitution pairs    : 3780


## 6. Verify All Outputs

In [14]:
print('=== Output checklist ===')
for path in [
    OUT_DIR + 'recipe_clean.csv',
    OUT_DIR + 'substitution.json',
]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) // 1024 if exists else 0
    print(f'  {"OK" if exists else "MISSING"} {path} ({size} KB)')

print(f'  OK  ChromaDB collection: {collection.count():,} recipes')
print()
print('Phase 1 (Core Data Layer) complete. Ready for Notebook 3.')

=== Output checklist ===
  OK ./output/recipe_clean.csv (571440 KB)
  OK ./output/substitution.json (128 KB)
  OK  ChromaDB collection: 225,044 recipes

Phase 1 (Core Data Layer) complete. Ready for Notebook 3.
